# Every experiment of the RBA paper, one cell each
Run the cells from top to bottom. Each training run saves `checkpoints/last_<row>_<task>_<subset>.pth` after every epoch, so a run that stops continues where it stopped when its cell is run again. To re-test a trained row without training it, add `--run eval` to its command.
Times are for one NVIDIA RTX 3070 Laptop GPU (8 GB): about 8 h per row on the 3 mm subset, about 14 h on the 6 mm subset, 1-2 h on ROSE-O.

In [ ]:
# CELL 0 - go to the repository folder and use this kernel's Python for every command
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PY = sys.executable
print('working folder:', os.getcwd())

In [ ]:
# CELL 1 - SEEM v1.0 + the two patched files + SEEM's weights (once; needs seem_patches/ filled in)
!{PY} setup_seem.py

In [ ]:
# CELL 2 - check the RBA module on its own (CPU, about a minute)
!{PY} rba_module.py

In [ ]:
# CELL 3 - data: OCTA-500 3 mm and 6 mm, ROSE-O -> the trainer's layout and the official splits
!{PY} prepare_data.py

In [ ]:
# CELL 4 - OCTA-500 3 mm, large vessels: every adapter of Table 1 (about 8 h each)
for row in ['H1_lora1_hq', 'C_convlora', 'B_lora', 'B1_lora1_ep100', 'AF_adaptformer', 'H_hq_only_ep100',
            'D_convlora_hq', 'B_lora_hq', 'A_decoder_only']:
    !{PY} train_rba.py --row {row} --task GT_LargeVessel --subset OCTA_3mm

In [ ]:
# CELL 5 - OCTA-500 3 mm, large vessels, all of SEEM frozen (Table 2)
for row in ['H1_frozendec', 'C_convlora_frozendec', 'B1_lora1_frozendec', 'H1_adapteronly']:
    !{PY} train_rba.py --row {row} --task GT_LargeVessel --subset OCTA_3mm

In [ ]:
# CELL 6 - OCTA-500 3 mm, veins (Tables 5-7)
for row in ['H1_lora1_hq', 'C_convlora', 'B1_lora1_ep100', 'H_hq_only_ep100', 'D_convlora_hq', 'A_decoder_only']:
    !{PY} train_rba.py --row {row} --task GT_Vein --subset OCTA_3mm

In [ ]:
# CELL 7 - OCTA-500 3 mm, arteries and FAZ (Table 5)
for row in ['H1_lora1_hq', 'C_convlora', 'H_hq_only_ep100']:
    !{PY} train_rba.py --row {row} --task GT_Artery --subset OCTA_3mm
for row in ['H1_lora1_hq', 'C_convlora']:
    !{PY} train_rba.py --row {row} --task GT_FAZ --subset OCTA_3mm

In [ ]:
# CELL 8 - the alternative read-out design RiDA and its ablation (large vessels and capillaries)
for row in ['R_rida', 'R_no_ridge', 'R_no_readout', 'R_no_encoder']:
    !{PY} train_rba.py --row {row} --task GT_LargeVessel --subset OCTA_3mm
for row in ['H1_lora1_hq', 'R_rida']:
    !{PY} train_rba.py --row {row} --task GT_Capillary --subset OCTA_3mm

In [ ]:
# CELL 9 - OCTA-500 6 mm (about 14 h each)
for task, rows in [('GT_LargeVessel', ['H1_lora1_hq', 'C_convlora']), ('GT_Vein', ['H1_lora1_hq', 'C_convlora']),
                   ('GT_Artery', ['H1_lora1_hq']), ('GT_FAZ', ['H1_lora1_hq'])]:
    for row in rows:
        !{PY} train_rba.py --row {row} --task {task} --subset OCTA_6mm

In [ ]:
# CELL 10 - ROSE-O vessels and FAZ (1-2 h each)
for task in ['GT_RoseVessel', 'GT_RoseFAZ']:
    for row in ['H1_lora1_hq', 'C_convlora']:
        !{PY} train_rba.py --row {row} --task {task} --subset ROSE_O

In [ ]:
# CELL 11 - tables: paired comparisons and claims (Tables 1, 5, 6), five metrics (Supplementary S1-S5)
!{PY} adapter_table.py
!{PY} evaluate_five_metrics.py

In [ ]:
# CELL 12 - where the gain lands (Table 8) and vessel identity (Table 7), no training
!{PY} train_rba.py --run calibre_check --task GT_LargeVessel --subset OCTA_3mm
!{PY} train_rba.py --run vein_check --task GT_Vein --subset OCTA_3mm